# Strands Agent with Datadog Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with Datadog observability integration. The implementation uses Amazon Bedrock Claude models and sends telemetry data to Datadog through OpenTelemetry (OTEL).

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Datadog**: Unified observability platform for monitoring, APM, logs, and traces
- **OpenTelemetry**: Industry-standard protocol for collecting and exporting telemetry data

## Architecture

The agent is containerized and deployed to AgentCore Runtime, which provides HTTP endpoints for invocation. Telemetry data flows from the Strands agent through OTEL exporters to Datadog for monitoring and debugging. The implementation disables AgentCore's default observability to use Datadog instead.

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- [Datadog](https://www.datadoghq.com/) account with API key
- Docker installed locally
- Access to Amazon Bedrock Claude models in us-west-2

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configure AWS Credentials

## Agent Implementation

The agent file (`strands_claude.py`) implements a travel agent with web search capabilities. Key configuration includes:
- **Module-level telemetry initialization**: StrandsTelemetry is initialized once when the module loads, setting up a global tracer provider
- **OTLP configuration via environment variables**: The agent reads `DD_API_KEY` and configures OTLP settings before initializing telemetry
- **Automatic trace export**: All agent invocations, tool calls, and LLM interactions are automatically traced and sent to Datadog

In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format=\"[%(levelname)s] %(message)s\")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv(\"AGENT_RUNTIME_LOG_LEVEL\", \"INFO\").upper())

# Configure OpenTelemetry for Datadog if API key is provided
# This MUST be done BEFORE initializing StrandsTelemetry
dd_api_key = os.getenv(\"DD_API_KEY\")
if dd_api_key:
    os.environ[\"OTEL_EXPORTER_OTLP_TRACES_PROTOCOL\"] = \"http/protobuf\"
    os.environ[\"OTEL_EXPORTER_OTLP_TRACES_ENDPOINT\"] = \"https://trace.agent.datadoghq.com/v1/traces\"
    os.environ[\"OTEL_EXPORTER_OTLP_TRACES_HEADERS\"] = f\"dd-api-key={dd_api_key},dd-otlp-source=datadog\"
    os.environ[\"OTEL_SEMCONV_STABILITY_OPT_IN\"] = \"gen_ai_latest_experimental\"
    logger.info(\"✓ Datadog OTLP environment variables configured\")
    
    # Initialize StrandsTelemetry at module level (sets up global tracer)
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    logger.info(\"✓ Datadog telemetry initialized with OTLP exporter\")
else:
    logger.warning(\"⚠ No DD_API_KEY provided, running without Datadog telemetry\")


@tool
def web_search(query: str) -> str:
    \"\"\"
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    \"\"\"
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f\"{i}. {result.get('title', 'No title')}\\n\"
                f\"   {result.get('body', 'No summary')}\\n\"
                f\"   Source: {result.get('href', 'No URL')}\\n\"
            )

        return \"\\n\".join(formatted_results) if formatted_results else \"No results found.\"

    except Exception as e:
        return f\"Error searching the web: {str(e)}\"


def get_bedrock_model():
    region = os.getenv(\"AWS_DEFAULT_REGION\", \"us-west-2\")
    model_id = os.getenv(\"BEDROCK_MODEL_ID\", \"global.anthropic.claude-haiku-4-5-20251001-v1:0\")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model


bedrock_model = get_bedrock_model()

system_prompt = \"\"\"You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details.\"\"\"

app = BedrockAgentCoreApp()


def initialize_agent():
    \"\"\"Initialize the agent (telemetry is already configured at module level).\"\"\"
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    \"\"\"
    Invoke the agent with a payload
    \"\"\"
    user_input = payload.get(\"prompt\")
    logger.info(\"[%s] User input: %s\", context.session_id, user_input)
    
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == \"__main__\":
    app.run()

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it configures AgentCore Observability by default so, to use Datadog, you need to remove configuration for AgentCore Observability as explained below:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_datadog_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

## Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

### Datadog Configuration

To send traces to Datadog, you only need:
- **Datadog API Key**: Get this from your Datadog account at Organization Settings → API Keys

The agent code (`strands_claude.py`) automatically configures all OTLP settings when `DD_API_KEY` is provided:
- **Endpoint**: `https://trace.agent.datadoghq.com/v1/traces` (US1 region)
- **Headers**: `dd-api-key={key},dd-otlp-source=datadog`
- **Protocol**: `http/protobuf`
- **Semantic Conventions**: `gen_ai_latest_experimental` (for LLM observability)

**For other Datadog regions**, update the endpoint in `strands_claude.py`:
- US1: `https://trace.agent.datadoghq.com/v1/traces` (default)
- US3: `https://trace.agent.us3.datadoghq.com/v1/traces`
- US5: `https://trace.agent.us5.datadoghq.com/v1/traces`
- EU1: `https://trace.agent.datadoghq.eu/v1/traces`
- AP1: `https://trace.agent.ap1.datadoghq.com/v1/traces`

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
# Datadog configuration
datadog_api_key = "<datadog-api-key>"  # Replace with your Datadog API key

# Note: OTLP configuration is handled in strands_claude.py
# The agent code will automatically configure the correct Datadog OTLP endpoint,
# headers, protocol, and semantic conventions when DD_API_KEY is provided.
# This follows the pattern from the working strands-todo-app reference implementation.

launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
        "DD_API_KEY": datadog_api_key,  # Datadog API key - agent will configure OTLP
        "DISABLE_ADOT_OBSERVABILITY": "true",  # Disable AgentCore's default observability
        "AWS_DEFAULT_REGION": "us-east-1",
    }
)
launch_result


## Check Deployment Status

Wait for the runtime to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to Paris. What are the must-visit places and local food I should try?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## View Traces in Datadog

To view the traces:
1. Go to your Datadog dashboard at https://app.datadoghq.com (or your regional site)
2. Navigate to **APM** → **Traces**
3. Look for traces from your service (service name will be derived from your agent)
4. You can also view:
   - **Service Map**: APM → Service Map to see service dependencies
   - **Service Catalog**: APM → Service Catalog for service overview
   - **Trace Explorer**: APM → Traces → Search to filter and analyze traces

The traces will include:
- Agent invocation details with full request/response context
- Tool calls (web search) with execution time
- Model interactions with latency and token usage
- Request/response payloads
- Distributed tracing across services
- Performance metrics and analytics

### Datadog APM Features

Datadog provides comprehensive observability:
- **Flame Graphs**: Visualize trace spans and identify bottlenecks
- **Service Dependencies**: Understand how services interact
- **Error Tracking**: Automatic error detection and grouping
- **Performance Monitoring**: Track latency, throughput, and errors
- **Custom Tags**: Filter and group traces by custom attributes
- **Alerting**: Set up monitors for performance degradation

**Placeholder for Datadog APM screenshot**

## Cleanup (Optional)

Clean up the deployed resources:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Datadog observability. The implementation demonstrates:
- Integration of Strands agents with AgentCore Runtime
- Configuration of OpenTelemetry to send traces to Datadog
- Proper initialization order to ensure telemetry configuration
- Invocation through both SDK and boto3 client

The agent is now running in a managed, scalable environment with full observability through Datadog. Datadog provides:
- Unified APM, logs, and infrastructure monitoring
- Real-time performance insights and anomaly detection
- Service dependency mapping and distributed tracing
- Custom dashboards and alerting capabilities
- Integration with 500+ technologies and services